# Operational Cost Modeling: CDMX Trolleybus Line 2

This notebook evaluates the operational cost of the Trolleybus Line 2 in Mexico City. It uses line integrals to calculate the cumulative cost along the route based on the geographic coordinates of the stations.

In [2]:
import numpy as np
import pandas as pd
import math

stations_df = pd.read_csv("data/coordinates.csv")
stations_df.head()

,Estación (Nodo i),y (Latitud),x (Longitud)
0,1. Pantitlán,19.416323,-99.073544
1,2. Ciudad Deportiva,19.408642,-99.090894
2,3. Velódromo,19.408365,-99.105037
3,4. Congreso de la Unión,19.409245,-99.121331
4,5. La Viga,19.413514,-99.127686


## 1. Trajectory Parameterization

The route is modeled as a sequence of linear segments. Each segment between two stations is parameterized using the variable `t` on the interval [0, 1].

In [3]:
for i in range(len(stations_df) - 1):
    x_current = stations_df.loc[i, "x (Longitud)"]
    y_current = stations_df.loc[i, "y (Latitud)"]
    x_next = stations_df.loc[i+1, "x (Longitud)"]
    y_next = stations_df.loc[i+1, "y (Latitud)"]
    
    delta_x = x_next - x_current
    delta_y = y_next - y_current

    print(f"Segment {i}:")
    print(f"x(t) = {x_current} + ({delta_x}) * t")
    print(f"y(t) = {y_current} + ({delta_y}) * t")

Segment 0:
x(t) = -99.07354354 + (-0.017350210000003585) * t
y(t) = 19.41632298 + (-0.007680549999999897) * t
Segment 1:
x(t) = -99.09089375 + (-0.014143369999999322) * t
y(t) = 19.40864243 + (-0.00027715999999955443) * t
Segment 2:
x(t) = -99.10503712 + (-0.01629361999999901) * t
y(t) = 19.40836527 + (0.0008795699999986084) * t
Segment 3:
x(t) = -99.12133074 + (-0.00635486000000185) * t
y(t) = 19.40924484 + (0.004268970000001815) * t
Segment 4:
x(t) = -99.1276856 + (-0.008835359999991965) * t
y(t) = 19.41351381 + (0.0007012799999976949) * t
Segment 5:
x(t) = -99.13652096 + (-0.011336090000000354) * t
y(t) = 19.41421509 + (0.0012952199999993752) * t
Segment 6:
x(t) = -99.14785705 + (-0.004637680000001865) * t
y(t) = 19.41551031 + (0.0003512300000032553) * t
Segment 7:
x(t) = -99.15249473 + (-0.012670700000001034) * t
y(t) = 19.41586154 + (-0.0003663600000010092) * t
Segment 8:
x(t) = -99.16516543 + (-0.010168179999993754) * t
y(t) = 19.41549518 + (0.005085539999999611) * t


## 2. Distance Calculation (Haversine Formula)

To evaluate the line integral accurately, angular coordinates must be converted to physical distances. The Haversine formula calculates the real differential arc length (`ds`) in kilometers.

In [4]:
def calculate_haversine_distance(lon1, lat1, lon2, lat2):
    R = 6371.0 # Earth radius in km
    lon1_rad, lat1_rad = math.radians(lon1), math.radians(lat1)
    lon2_rad, lat2_rad = math.radians(lon2), math.radians(lat2)
    
    dlon = lon2_rad - lon1_rad
    dlat = lat2_rad - lat1_rad
    
    a = math.sin(dlat / 2)**2 + math.cos(lat1_rad) * math.cos(lat2_rad) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

## 3. Cost Density Function and Line Integral

The scalar field defines the cost per kilometer. It incorporates the base cost and adds penalties for complex topography and elevated infrastructure. The total cost is evaluated through the line integral over the entire trajectory.

In [5]:
# Financial parameters (MXN)
C0 = 17.73      # Base cost per km
ALPHA = 0.20    # Penalty for tight curves
BETA = 0.005    # Penalty for elevated infrastructure

# Assign complexity factors to specific segments
stations_df['p'] = 0  
stations_df['h'] = 0  
stations_df.loc[stations_df['Estación (Nodo i)'].str.contains('Velódromo', na=False), 'p'] = 1 
stations_df.loc[stations_df['Estación (Nodo i)'].str.contains('Ciudad Deportiva|5 de Febrero', na=False), 'h'] = 1 

total_route_cost = 0.0

print(f"{'Segment':<10} | {'Length (km)':<15} | {'Cost/km':<12} | {'Subtotal':<12}")
print("-" * 55)

for i in range(len(stations_df) - 1):
    lon1, lat1 = stations_df.loc[i, "x (Longitud)"], stations_df.loc[i, "y (Latitud)"]
    lon2, lat2 = stations_df.loc[i+1, "x (Longitud)"], stations_df.loc[i+1, "y (Latitud)"]
    
    p_segment = stations_df.loc[i, "p"]
    h_segment = stations_df.loc[i, "h"]
    
    ds = calculate_haversine_distance(lon1, lat1, lon2, lat2)
    
    cost_density = C0 * (1 + (ALPHA * p_segment) + (BETA * h_segment))
    subtotal = cost_density * ds
    total_route_cost += subtotal
    
    print(f"{i:<10} | {ds:<15.4f} | ${cost_density:<11.2f} | ${subtotal:<11.2f}")

print("-" * 55)
print(f"Total estimated cost for the route: ${total_route_cost:.2f} MXN")

Segment    | Length (km)     | Cost/km      | Subtotal    
-------------------------------------------------------
0          | 2.0100          | $17.73       | $35.64      
1          | 1.4836          | $17.82       | $26.44      
2          | 1.7116          | $21.28       | $36.42      
3          | 0.8182          | $17.73       | $14.51      
4          | 0.9299          | $17.73       | $16.49      
5          | 1.1975          | $17.82       | $21.34      
6          | 0.4879          | $17.73       | $8.65       
7          | 1.3294          | $17.73       | $23.57      
8          | 1.2070          | $17.73       | $21.40      
-------------------------------------------------------
Total estimated cost for the route: $204.44 MXN


## 4. Convergence Analysis (Riemann Sums)

To verify numerical stability, the integral approximation is tested using dyadic partitions.

In [6]:
def dyadic_riemann_sum(k, df, C0, ALPHA, BETA):
    partitions = 2**k
    estimated_cost = 0.0
    
    for i in range(len(df) - 1):
        lon1, lat1 = df.loc[i, "x (Longitud)"], df.loc[i, "y (Latitud)"]
        lon2, lat2 = df.loc[i+1, "x (Longitud)"], df.loc[i+1, "y (Latitud)"]
        p_segment, h_segment = df.loc[i, "p"], df.loc[i, "h"]
        
        ds_total = calculate_haversine_distance(lon1, lat1, lon2, lat2)
        ds_partition = ds_total / partitions
        
        cost_density = C0 * (1 + (ALPHA * p_segment) + (BETA * h_segment))
        
        for j in range(partitions):
            estimated_cost += cost_density * ds_partition
            
    return estimated_cost

print("--- DYADIC CONVERGENCE ANALYSIS ---")
k_values = [1, 5, 10, 12]

for k in k_values:
    result_k = dyadic_riemann_sum(k, stations_df, C0, ALPHA, BETA)
    partitions_per_segment = 2**k
    print(f"Range k={k:<2} | Partitions/segment: {partitions_per_segment:<5} | Estimated Cost: ${result_k:.2f} MXN")

--- DYADIC CONVERGENCE ANALYSIS ---
Range k=1  | Partitions/segment: 2     | Estimated Cost: $204.44 MXN
Range k=5  | Partitions/segment: 32    | Estimated Cost: $204.44 MXN
Range k=10 | Partitions/segment: 1024  | Estimated Cost: $204.44 MXN
Range k=12 | Partitions/segment: 4096  | Estimated Cost: $204.44 MXN
